In [ ]:
from datetime import datetime
import numpy as np
import pandas as pd
import yaml as yml
import pycountry
from plotly import express as px
import json
from plotly.subplots import make_subplots
from plotly import graph_objects as go
from IPython.display import Markdown
from matplotlib import pyplot as plt

from emu_renewal.inputs import BASE_PATH, DATA_PATH, get_standard_priors
from emu_renewal.inputs import get_worldbank_national_pop, get_country_mobility, get_standard_targets, get_euro_var_inputs

In [ ]:
# Standard inputs
analysis_start = datetime(2020, 9, 1)
analysis_end = datetime(2021, 12, 31)
init_duration = 50
var_target_start_date = datetime(2021, 1, 1)
var_target_end_date = datetime(2024, 4, 1)
seed_duration = 10
date_format = "%d/%m/%Y"

In [ ]:
initial_countries = json.load(open(DATA_PATH / f"config/countries.json", "r"))
all_countries = initial_countries["admissions"] + initial_countries["occupancy"]

In [ ]:
cases_fig, cases_axes = plt.subplots(5, 4, figsize=[12, 10], sharex=True)
cases_fig.suptitle("cases")
hosp_fig, hosp_axes = plt.subplots(5, 4, figsize=[12, 10], sharex=True)
hosp_fig.suptitle("hospitalisation (admissions black, occupancy red)")
deaths_fig, deaths_axes = plt.subplots(5, 4, figsize=[12, 10], sharex=True)
deaths_fig.suptitle("deaths")
sero_fig, sero_axes = plt.subplots(5, 4, figsize=[12, 10], sharex=True)
sero_fig.suptitle("seroprevalence")
mob_fig, mob_axes = plt.subplots(5, 4, figsize=[12, 10], sharex=True)
mob_fig.suptitle("mobility")
path = BASE_PATH / "preview_plots"
text = ""
flat_cases_axes = cases_axes.ravel()
flat_hosp_axes = hosp_axes.ravel()
flat_deaths_axes = deaths_axes.ravel()
flat_sero_axes = sero_axes.ravel()
flat_mob_axes = mob_axes.ravel()
for c, country in enumerate(all_countries):
    iso3 = pycountry.countries.lookup(country).alpha_3
    country_name = pycountry.countries.lookup(country).name
    pop = get_worldbank_national_pop(iso3)
    hosp_indicator, hosp_colour = ("Weekly new hospital admissions", "black") if country in initial_countries["admissions"] else ("Daily hospital occupancy", "red")
    cases_target, hosp_target, deaths_target, seroprev_target, init_data = get_standard_targets(
        country, analysis_start, analysis_end, init_duration, hosp_indicator,
    )
    # strains = ["eu", "alpha"]
    # var_target, seed_times = get_euro_var_inputs(country, strains, analysis_start, seed_duration, var_target_start_date, var_target_end_date)

    text += f"{country}\nISO3: {iso3}\n"
    text += f"population: {pop / 1e6} million\n"
    # text += f"alpha seed start: {datetime.strftime(seed_times[0][0], date_format)}\n"
    # text += f"alpha seed end: {datetime.strftime(seed_times[0][1], date_format)}\n"

    case_ax = flat_cases_axes[c]
    case_ax.plot(cases_target.index, cases_target, linewidth=0, marker=".")
    case_ax.set_title(country_name)
    plt.setp(case_ax.xaxis.get_majorticklabels(), rotation=70)
    hosp_ax = flat_hosp_axes[c]
    hosp_ax.plot(hosp_target.index, hosp_target, linewidth=0, marker=".", color=hosp_colour)
    hosp_ax.set_title(country_name)
    plt.setp(hosp_ax.xaxis.get_majorticklabels(), rotation=70)
    death_ax = flat_deaths_axes[c]
    death_ax.plot(deaths_target.index, deaths_target, linewidth=0, marker=".")
    death_ax.set_title(country_name)
    plt.setp(death_ax.xaxis.get_majorticklabels(), rotation=70)
    sero_ax = flat_sero_axes[c]
    sero_ax.plot(seroprev_target.index, seroprev_target, linewidth=0, marker=".")
    sero_ax.set_title(country_name)
    plt.setp(sero_ax.xaxis.get_majorticklabels(), rotation=70)
    # var_plot = px.scatter(var_target, title=f"variant proportions {country_name}")
    # var_plot.write_image(path / f"var_plot_{country}.png")

    mob_ax = flat_mob_axes[c]
    mob_ax.set_title(country_name)
    plt.setp(mob_ax.xaxis.get_majorticklabels(), rotation=70)
    try:
        mobility = get_country_mobility(iso3)
        mob_ax.plot(mobility.index, mobility)
    except:
        pass

cases_fig.tight_layout()
hosp_fig.tight_layout()
deaths_fig.tight_layout()
sero_fig.tight_layout()
mob_fig.tight_layout()